# PRISM steps 5-10: the schema wizard

The package version of the old website's Step 0. `prism.start` asks the AI (steps 2-4), then you go through steps 5-10, confirming or correcting each proposal. `s.todo()` always lists what is still open, with the command to close it. Every decision records who made it (computed, AI confirmed, AI corrected, set by you), and `s.log` keeps every action.

The cells below use typical answers as **examples**: read each output first and change the arguments to match your data.

In [ ]:
from google.colab import userdata, drive
token = userdata.get("GITHUB_TOKEN")
!pip install -q --force-reinstall --no-deps "git+https://{token}@github.com/Isra-inventor/PRISM.git@claude/prism-data-input-recognition-9adu1r"
import prism
print("PRISM", prism.__version__)
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/PRISM_data"
OUT = "/content/drive/MyDrive/PRISM_out"

In [ ]:
table = prism.read(f"{DATA}/SomaExpr_common.csv")
s = prism.start(table)      # asks for the Gemini key once if it is not in Secrets
s

## Step 5: groups

Read the proposal. Correct what is wrong, e.g. a numeric annotation (Kd, LoD, CV, reference RFU) proposed as a value:
```python
s.set("g7", role="feature_annotation", label="dissociation constant Kd")
s.split("g9", ["Normal Plasma RFU"], role="feature_annotation", label="reference plasma RFU")
s.merge("g10", "g11")
s.set_layout("samples_in_columns")
```
Then confirm what is left. `s.show("g3")` gives every column of a group, the AI's evidence and the facts.

In [ ]:
s.groups()

In [ ]:
# s.set("g7", role="feature_annotation", label="...")   # your corrections here
s.confirm_all()
s.todo()

## Step 6: protein ID and flagged rows

Flagged rows are kept and labelled, never dropped. `s.values_of("Type")` shows a column's values to choose from.

In [ ]:
display(s.feature_id())
s.flags()

In [ ]:
# s.flag_rows("Type", values=["Hybridization Control Elution", "Spuriomer", "Non-Biotin"])
# s.flag_rows("Organism", values=["Mouse", "Rat"])
s.confirm_feature_id()
s.confirm_flags()
s.flags()

## Step 7: values and samples

If a block holds zeros, say what they mean: `s.set_block("A1B1", zero_means="not_detected")` (or `"zero"`). Sample types: the AI sees the sample **names** only.

In [ ]:
display(s.blocks())
display(s.assays_table())
s.matrix("A1B1").iloc[:5, :5]

In [ ]:
s.confirm_values()
s.propose_sample_types()

In [ ]:
# s.set_sample_type("Calib*", "calibrator")      # corrections: a pattern or a list of names
s.confirm_samples()
s.samples()["sample_type"].value_counts()

## Step 8: sample information and design

If you have a sample-information file (patient, visit, plate, group ...), add it. Otherwise skip to the design cell.

In [ ]:
# s.add_sample_info(f"{DATA}/sample_info.csv")
# s.sample_info()
# s.set_info_column("Visit", audit_kind="timepoint")
# s.confirm_sample_info()
s.design()

If subject and time are only in the sample names, `s.name_patterns()` shows how the names split; choose a rule:
```python
s.set_design(derive={"delimiter": "_", "occurrence": "last", "left": "subject", "right": "time"}, time_unit="weeks")
```
With subject and time columns: `s.set_design(subject="Patient", time="Visit", time_unit="weeks")`. No design information: just confirm.

In [ ]:
display(s.name_patterns())
# s.set_design(...)
s.confirm_design()
s.design()

## Step 9: processing history

The hints come from the numbers; only you answer (yes / no / not_sure).

In [ ]:
s.history()

In [ ]:
s.answer_history(normalized="not_sure", log_transformed="no", imputed="not_sure", batch_corrected="not_sure",
                 features_or_samples_removed_before_upload="not_sure")

## Step 10: review and save

In [ ]:
s.review()

In [ ]:
s.save(f"{OUT}/soma")

Next time, reuse the saved schema instead of answering again:

In [ ]:
s2 = prism.load(f"{OUT}/soma/schema.json", prism.read(f"{DATA}/SomaExpr_common.csv"))
s2